# SpyCloud Incident Triage Notebook**Purpose:** Guide SOC analysts through a comprehensive investigation of SpyCloud-triggered incidents in Microsoft Sentinel.This notebook correlates SpyCloud breach/malware exposure data with Sentinel telemetry to provide full incident context, identity analysis, device investigation, network correlation, and remediation tracking.---

## Section 1: Setup & ConfigurationInstall and import required libraries, authenticate to your Sentinel workspace, and set investigation parameters.

In [ ]:
# Install dependencies if needed# %pip install msticpy[all] pandas matplotlib plotly networkx azure-identityimport warningswarnings.filterwarnings("ignore")import pandas as pdimport matplotlib.pyplot as pltimport plotly.express as pximport plotly.graph_objects as gofrom plotly.subplots import make_subplotsimport networkx as nxfrom datetime import datetime, timedeltafrom IPython.display import display, HTML, Markdownimport msticpy as mpmp.init_notebook(namespace=globals(), verbosity=0)

In [ ]:
# Connect to Microsoft Sentinel workspace# Update with your workspace details or use msticpy configqry_prov = QueryProvider("MSSentinel")qry_prov.connect(    connection_str="loganalytics://code().tenant('YOUR_TENANT_ID')"    ".workspace('YOUR_WORKSPACE_ID')")print("Connected to Sentinel workspace successfully.")

In [ ]:
# === Investigation Parameters ===# Set the incident ID to investigateINCIDENT_ID = "INCIDENT-ID-HERE"  # e.g., "12345"# Lookback period for queriesLOOKBACK_DAYS = 30START_TIME = datetime.utcnow() - timedelta(days=LOOKBACK_DAYS)END_TIME = datetime.utcnow()# These will be populated dynamically from incident dataTARGET_USER = NoneTARGET_DEVICES = []TARGET_IPS = []print(f"Investigating Incident: {INCIDENT_ID}")print(f"Time range: {START_TIME.isoformat()} to {END_TIME.isoformat()}")

## Section 2: Incident ContextRetrieve incident details, associated entities, and build an incident timeline.

In [ ]:
# Query incident details from SecurityIncident tableincident_query = f"""SecurityIncident| where IncidentNumber == {INCIDENT_ID}| project    IncidentNumber,    Title,    Description,    Severity,    Status,    Classification,    Owner,    CreatedTime,    LastModifiedTime,    ClosedTime,    ProviderName,    AlertIds,    BookmarkIds,    Comments,    Labels,    AdditionalData| take 1"""incident_df = qry_prov.exec_query(incident_query)if incident_df is not None and not incident_df.empty:    inc = incident_df.iloc[0]    display(Markdown(f"### Incident: {inc['Title']}"))    display(Markdown(f"- **Severity:** {inc['Severity']}"))    display(Markdown(f"- **Status:** {inc['Status']}"))    display(Markdown(f"- **Created:** {inc['CreatedTime']}"))    display(Markdown(f"- **Provider:** {inc['ProviderName']}"))    display(Markdown(f"- **Description:** {inc['Description']}"))else:    print(f"No incident found with ID: {INCIDENT_ID}")

In [ ]:
# Get all entities (users, devices, IPs) associated with the incidententities_query = f"""SecurityIncident| where IncidentNumber == {INCIDENT_ID}| mv-expand AlertIds| extend AlertId = tostring(AlertIds)| join kind=inner (    SecurityAlert    | mv-expand Entities    | extend Entity = parse_json(Entities)    | project AlertId = SystemAlertId, Entity,              EntityType = tostring(Entity.Type)) on AlertId| summarize    Users = make_set_if(tostring(Entity.Name), EntityType == "account"),    Hosts = make_set_if(tostring(Entity.HostName), EntityType == "host"),    IPs = make_set_if(tostring(Entity.Address), EntityType == "ip"),    URLs = make_set_if(tostring(Entity.Url), EntityType == "url"),    FileHashes = make_set_if(tostring(Entity.Value), EntityType == "filehash")"""entities_df = qry_prov.exec_query(entities_query)if entities_df is not None and not entities_df.empty:    ent = entities_df.iloc[0]    TARGET_USER = ent['Users'][0] if ent['Users'] else None    TARGET_DEVICES = list(ent['Hosts']) if ent['Hosts'] else []    TARGET_IPS = list(ent['IPs']) if ent['IPs'] else []    display(Markdown("### Incident Entities"))    display(Markdown(f"- **Users:** {', '.join(ent['Users'])}"))    display(Markdown(f"- **Hosts:** {', '.join(ent['Hosts'])}"))    display(Markdown(f"- **IPs:** {', '.join(ent['IPs'])}"))    print(f"\nTarget user set to: {TARGET_USER}")else:    print("No entities found. Set TARGET_USER manually above.")

In [ ]:
# Display incident timeline: alerts, bookmarks, commentstimeline_query = f"""SecurityIncident| where IncidentNumber == {INCIDENT_ID}| mv-expand AlertIds| extend AlertId = tostring(AlertIds)| join kind=inner (    SecurityAlert    | project AlertId = SystemAlertId, AlertName = DisplayName,              AlertSeverity = AlertSeverity, AlertTime = TimeGenerated,              Tactics, Techniques) on AlertId| project AlertTime, AlertName, AlertSeverity, Tactics, Techniques| order by AlertTime asc"""timeline_df = qry_prov.exec_query(timeline_query)if timeline_df is not None and not timeline_df.empty:    fig = px.timeline(        timeline_df.assign(            Start=pd.to_datetime(timeline_df['AlertTime']),            End=pd.to_datetime(timeline_df['AlertTime']) + timedelta(minutes=30)        ),        x_start="Start", x_end="End", y="AlertName",        color="AlertSeverity",        title="Incident Alert Timeline",        color_discrete_map={"High": "red", "Medium": "orange", "Low": "yellow", "Informational": "blue"}    )    fig.update_layout(height=400)    fig.show()else:    print("No alerts found for this incident.")

## Section 3: SpyCloud Exposure AnalysisAnalyze SpyCloud breach and malware data for the affected user to understand the scope and severity of credential exposures.

In [ ]:
# Query SpyCloudBreachWatchlist_CL for affected userif TARGET_USER:    breach_query = f"""    SpyCloudBreachWatchlist_CL    | where email_s contains "{TARGET_USER}" or username_s contains "{TARGET_USER}"    | project        TimeGenerated,        email_s,        username_s,        domain_s,        breach_title_s,        spycloud_publish_date_t,        password_type_s,        severity_d,        source_id_d,        infected_machine_id_s,        infected_path_s,        target_url_s,        ip_address_s,        document_id_s    | order by TimeGenerated desc    """    breach_df = qry_prov.exec_query(breach_query)    if breach_df is not None and not breach_df.empty:        display(Markdown(f"### SpyCloud Exposures for {TARGET_USER}"))        display(Markdown(f"**Total records:** {len(breach_df)}"))        display(breach_df.head(20))    else:        print(f"No SpyCloud breach data found for {TARGET_USER}")else:    print("TARGET_USER not set. Please set it manually in Section 1.")

In [ ]:
# Query SpyCloudCompassData_CL for related recordsif TARGET_USER:    compass_query = f"""    SpyCloudCompassData_CL    | where email_s contains "{TARGET_USER}" or username_s contains "{TARGET_USER}"    | project        TimeGenerated,        email_s,        infected_machine_id_s,        infected_time_t,        malware_family_s,        target_url_s,        password_type_s,        severity_d,        source_id_d,        ip_address_s,        cc_bin_s,        log_id_s,        document_id_s    | order by TimeGenerated desc    """    compass_df = qry_prov.exec_query(compass_query)    if compass_df is not None and not compass_df.empty:        display(Markdown(f"### SpyCloud Compass Data for {TARGET_USER}"))        display(Markdown(f"**Total Compass records:** {len(compass_df)}"))        display(compass_df.head(20))    else:        print(f"No Compass data found for {TARGET_USER}")

In [ ]:
# Exposure history timeline (plotly chart)if breach_df is not None and not breach_df.empty:    timeline_data = breach_df.copy()    timeline_data['date'] = pd.to_datetime(timeline_data['spycloud_publish_date_t'])    exposure_timeline = timeline_data.groupby(        [pd.Grouper(key='date', freq='M'), 'password_type_s']    ).size().reset_index(name='count')    fig = px.bar(        exposure_timeline,        x='date', y='count', color='password_type_s',        title=f"Exposure History Timeline for {TARGET_USER}",        labels={'date': 'Date', 'count': 'Exposures', 'password_type_s': 'Password Type'},        barmode='stack'    )    fig.update_layout(height=450, xaxis_title="Month", yaxis_title="Number of Exposures")    fig.show()

In [ ]:
# Password type breakdown (pie chart)if breach_df is not None and not breach_df.empty:    pwd_types = breach_df['password_type_s'].value_counts().reset_index()    pwd_types.columns = ['Password Type', 'Count']    fig = px.pie(        pwd_types, names='Password Type', values='Count',        title="Password Type Distribution",        color_discrete_sequence=px.colors.qualitative.Set2,        hole=0.3    )    fig.update_traces(textposition='inside', textinfo='percent+label')    fig.show()    # Severity distribution    sev_dist = breach_df['severity_d'].value_counts().sort_index().reset_index()    sev_dist.columns = ['Severity', 'Count']    fig2 = px.bar(        sev_dist, x='Severity', y='Count',        title="Severity Distribution of Exposures",        color='Severity',        color_continuous_scale='RdYlGn_r'    )    fig2.show()

In [ ]:
# Related breach catalog entriesif breach_df is not None and not breach_df.empty:    source_ids = breach_df['source_id_d'].dropna().unique().tolist()    source_ids_str = ",".join([str(int(s)) for s in source_ids])    catalog_query = f"""    SpyCloudBreachCatalog_CL    | where source_id_d in ({source_ids_str})    | project        source_id_d,        title_s,        description_s,        type_s,        num_records_d,        spycloud_publish_date_t,        acquisition_date_t,        confidence_d,        site_s    | distinct *    """    catalog_df = qry_prov.exec_query(catalog_query)    if catalog_df is not None and not catalog_df.empty:        display(Markdown("### Related Breach Catalog Entries"))        display(catalog_df)    else:        print("No breach catalog entries found for these source IDs.")

## Section 4: Identity CorrelationCross-reference SpyCloud exposure data with Azure AD sign-in activity, risk events, and MFA configuration changes.

In [ ]:
# Query SigninLogs for affected user (last 30 days)if TARGET_USER:    signin_query = f"""    SigninLogs    | where TimeGenerated >= ago(30d)    | where UserPrincipalName contains "{TARGET_USER}"    | project        TimeGenerated,        UserPrincipalName,        AppDisplayName,        IPAddress,        Location = strcat(LocationDetails.city, ", ",                          LocationDetails.state, ", ",                          LocationDetails.countryOrRegion),        Latitude = todouble(LocationDetails.geoCoordinates.latitude),        Longitude = todouble(LocationDetails.geoCoordinates.longitude),        ResultType,        ResultDescription,        RiskLevelDuringSignIn,        RiskState,        ConditionalAccessStatus,        MfaDetail,        DeviceDetail,        ClientAppUsed,        AuthenticationRequirement    | order by TimeGenerated desc    """    signin_df = qry_prov.exec_query(signin_query)    if signin_df is not None and not signin_df.empty:        display(Markdown(f"### Sign-in Activity for {TARGET_USER}"))        display(Markdown(f"**Total sign-ins (30 days):** {len(signin_df)}"))        display(Markdown(f"**Unique IPs:** {signin_df['IPAddress'].nunique()}"))        display(Markdown(f"**Unique Locations:** {signin_df['Location'].nunique()}"))        display(signin_df.head(20))    else:        print(f"No sign-in logs found for {TARGET_USER}")

In [ ]:
# Check for risky sign-ins (AADUserRiskEvents)if TARGET_USER:    risk_query = f"""    AADUserRiskEvents    | where TimeGenerated >= ago(30d)    | where UserPrincipalName contains "{TARGET_USER}"    | project        TimeGenerated,        UserPrincipalName,        RiskEventType,        RiskLevel,        RiskState,        RiskDetail,        IpAddress,        Location,        DetectedDateTime,        LastUpdatedDateTime,        Source    | order by TimeGenerated desc    """    risk_df = qry_prov.exec_query(risk_query)    if risk_df is not None and not risk_df.empty:        display(Markdown("### User Risk Events"))        display(risk_df)        # Risk event type breakdown        fig = px.bar(            risk_df['RiskEventType'].value_counts().reset_index(),            x='index', y='RiskEventType',            title="Risk Event Types",            labels={'index': 'Event Type', 'RiskEventType': 'Count'}        )        fig.show()    else:        print("No risk events found.")

In [ ]:
# Check Conditional Access evaluation resultsif signin_df is not None and not signin_df.empty:    ca_summary = signin_df['ConditionalAccessStatus'].value_counts().reset_index()    ca_summary.columns = ['CA Status', 'Count']    display(Markdown("### Conditional Access Evaluation Summary"))    display(ca_summary)    fig = px.pie(        ca_summary, names='CA Status', values='Count',        title="Conditional Access Outcomes",        color_discrete_map={'success': 'green', 'failure': 'red', 'notApplied': 'gray'}    )    fig.show()

In [ ]:
# Check MFA registration/changes (AuditLogs)if TARGET_USER:    mfa_query = f"""    AuditLogs    | where TimeGenerated >= ago(30d)    | where OperationName has_any (        "User registered security info",        "User deleted security info",        "User changed default security info",        "Admin registered security info",        "User registered all required security info",        "User started security info registration"    )    | extend Target = tostring(TargetResources[0].userPrincipalName)    | where Target contains "{TARGET_USER}"    | project        TimeGenerated,        OperationName,        Target,        InitiatedBy = tostring(InitiatedBy.user.userPrincipalName),        Result,        ResultReason,        AdditionalDetails    | order by TimeGenerated desc    """    mfa_df = qry_prov.exec_query(mfa_query)    if mfa_df is not None and not mfa_df.empty:        display(Markdown("### MFA Registration/Changes"))        display(mfa_df)    else:        print("No MFA registration changes found.")

In [ ]:
# Display sign-in map (geographic visualization)if signin_df is not None and not signin_df.empty:    geo_df = signin_df.dropna(subset=['Latitude', 'Longitude'])    if not geo_df.empty:        geo_summary = geo_df.groupby(['Location', 'Latitude', 'Longitude']).agg(            SignInCount=('TimeGenerated', 'count'),            UniqueApps=('AppDisplayName', 'nunique'),            LastSeen=('TimeGenerated', 'max')        ).reset_index()        fig = px.scatter_geo(            geo_summary,            lat='Latitude', lon='Longitude',            size='SignInCount',            hover_name='Location',            hover_data=['SignInCount', 'UniqueApps', 'LastSeen'],            title=f"Sign-in Locations for {TARGET_USER}",            projection='natural earth',            color='SignInCount',            color_continuous_scale='Reds'        )        fig.update_layout(height=500)        fig.show()    else:        print("No geolocation data available in sign-in logs.")

In [ ]:
# Timeline of sign-ins vs exposure datesif signin_df is not None and not signin_df.empty and breach_df is not None and not breach_df.empty:    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.1,                        subplot_titles=("Sign-in Activity", "SpyCloud Exposures"))    # Sign-ins per day    signin_daily = signin_df.groupby(        pd.to_datetime(signin_df['TimeGenerated']).dt.date    ).size().reset_index(name='count')    signin_daily.columns = ['date', 'count']    fig.add_trace(        go.Bar(x=signin_daily['date'], y=signin_daily['count'],               name='Sign-ins', marker_color='steelblue'),        row=1, col=1    )    # Exposures per date    exposure_dates = breach_df.copy()    exposure_dates['date'] = pd.to_datetime(exposure_dates['spycloud_publish_date_t']).dt.date    exposure_daily = exposure_dates.groupby('date').size().reset_index(name='count')    fig.add_trace(        go.Bar(x=exposure_daily['date'], y=exposure_daily['count'],               name='Exposures', marker_color='crimson'),        row=2, col=1    )    fig.update_layout(height=600, title_text="Sign-in Activity vs Exposure Timeline")    fig.show()

## Section 5: Device InvestigationInvestigate devices associated with SpyCloud malware infections and cross-reference with Microsoft Defender for Endpoint telemetry.

In [ ]:
# Query SpyCloudCompassDevices_CL for infected devicesdevice_query = """SpyCloudCompassDevices_CL| where infected_machine_id_s != ""| project    TimeGenerated,    infected_machine_id_s,    infected_path_s,    infected_time_t,    hostname_s,    ip_address_s,    malware_family_s,    os_s,    user_name_s,    email_s| order by TimeGenerated desc"""if TARGET_USER:    device_query = f"""    SpyCloudCompassDevices_CL    | where email_s contains "{TARGET_USER}" or user_name_s contains "{TARGET_USER}"    | project        TimeGenerated,        infected_machine_id_s,        infected_path_s,        infected_time_t,        hostname_s,        ip_address_s,        malware_family_s,        os_s,        user_name_s,        email_s    | order by TimeGenerated desc    """devices_df = qry_prov.exec_query(device_query)if devices_df is not None and not devices_df.empty:    display(Markdown("### Infected Devices"))    display(devices_df)    TARGET_DEVICES.extend(devices_df['hostname_s'].dropna().unique().tolist())    TARGET_DEVICES = list(set(TARGET_DEVICES))    print(f"\nTracking devices: {TARGET_DEVICES}")else:    print("No infected devices found in SpyCloud Compass data.")

In [ ]:
# Cross-reference with Spycloud_MDE_Logs_CLif TARGET_USER or TARGET_DEVICES:    where_clause = []    if TARGET_USER:        where_clause.append(f'AccountName_s contains "{TARGET_USER}"')    if TARGET_DEVICES:        device_filter = " or ".join([f'DeviceName_s contains "{d}"' for d in TARGET_DEVICES])        where_clause.append(f"({device_filter})")    mde_query = f"""    Spycloud_MDE_Logs_CL    | where {" or ".join(where_clause)}    | project        TimeGenerated,        DeviceName_s,        AccountName_s,        ActionType_s,        RemediationStatus_s,        ActionStatus,        ScanStatus_s,        ThreatName_s,        Severity_s    | order by TimeGenerated desc    """    mde_logs_df = qry_prov.exec_query(mde_query)    if mde_logs_df is not None and not mde_logs_df.empty:        display(Markdown("### MDE Remediation Logs"))        display(mde_logs_df)    else:        print("No MDE remediation logs found.")

In [ ]:
# Check device compliance in Intune (DeviceInfo)if TARGET_DEVICES:    devices_str = " or ".join([f'DeviceName contains "{d}"' for d in TARGET_DEVICES])    compliance_query = f"""    DeviceInfo    | where TimeGenerated >= ago(30d)    | where {devices_str}    | summarize arg_max(TimeGenerated, *) by DeviceId    | project        DeviceName,        OSPlatform,        OSVersion,        JoinType,        MachineGroup,        DeviceCategory = MachineGroup,        IsAzureADJoined,        LoggedOnUsers,        PublicIP,        SensorHealthState,        OnboardingStatus,        ExposureLevel,        TimeGenerated    """    compliance_df = qry_prov.exec_query(compliance_query)    if compliance_df is not None and not compliance_df.empty:        display(Markdown("### Device Compliance Status"))        display(compliance_df)    else:        print("No device compliance info found.")

In [ ]:
# Check for malware detections on devices (DeviceEvents)if TARGET_DEVICES:    devices_str = " or ".join([f'DeviceName contains "{d}"' for d in TARGET_DEVICES])    malware_query = f"""    DeviceEvents    | where TimeGenerated >= ago(30d)    | where {devices_str}    | where ActionType in (        "AntivirusDetection", "ExploitGuardNonAuditNetworkProtection",        "SmartScreenAppWarning", "SmartScreenUrlWarning",        "FirewallInboundConnectionBlocked"    )    | project        TimeGenerated, DeviceName, ActionType,        FileName, FolderPath, SHA256 = SHA256,        AdditionalFields    | order by TimeGenerated desc    """    malware_df = qry_prov.exec_query(malware_query)    if malware_df is not None and not malware_df.empty:        display(Markdown("### Malware Detections"))        display(malware_df)    else:        print("No malware detections found on tracked devices.")

In [ ]:
# Device network activity (DeviceNetworkEvents)if TARGET_DEVICES:    devices_str = " or ".join([f'DeviceName contains "{d}"' for d in TARGET_DEVICES])    network_query = f"""    DeviceNetworkEvents    | where TimeGenerated >= ago(7d)    | where {devices_str}    | summarize        ConnectionCount = count(),        UniqueRemoteIPs = dcount(RemoteIP),        UniqueRemotePorts = dcount(RemotePort),        BytesSent = sum(todouble(AdditionalFields.BytesSent)),        BytesReceived = sum(todouble(AdditionalFields.BytesReceived))      by DeviceName, RemoteIP, RemotePort, RemoteUrl, ActionType    | order by ConnectionCount desc    | take 50    """    net_df = qry_prov.exec_query(network_query)    if net_df is not None and not net_df.empty:        display(Markdown("### Top Network Connections (Last 7 Days)"))        display(net_df)    else:        print("No network events found.")

In [ ]:
# Process execution timelineif TARGET_DEVICES:    devices_str = " or ".join([f'DeviceName contains "{d}"' for d in TARGET_DEVICES])    proc_query = f"""    DeviceProcessEvents    | where TimeGenerated >= ago(7d)    | where {devices_str}    | project        TimeGenerated, DeviceName, FileName,        ProcessCommandLine, FolderPath,        AccountName, InitiatingProcessFileName,        InitiatingProcessCommandLine    | order by TimeGenerated desc    | take 100    """    proc_df = qry_prov.exec_query(proc_query)    if proc_df is not None and not proc_df.empty:        display(Markdown("### Process Execution Timeline"))        # Show top processes by frequency        top_procs = proc_df['FileName'].value_counts().head(15).reset_index()        top_procs.columns = ['Process', 'Count']        fig = px.bar(top_procs, x='Process', y='Count',                     title="Top 15 Processes Executed on Infected Devices")        fig.show()        display(proc_df.head(30))    else:        print("No process events found.")

## Section 6: Network & Firewall CorrelationCorrelate SpyCloud infection data with firewall, DNS, and network security appliance logs.

In [ ]:
# Query CommonSecurityLog for SpyCloud-related IPsif TARGET_IPS:    ips_str = " or ".join([f'SourceIP == "{ip}" or DestinationIP == "{ip}"' for ip in TARGET_IPS])    csl_query = f"""    CommonSecurityLog    | where TimeGenerated >= ago(30d)    | where {ips_str}    | project        TimeGenerated, DeviceVendor, DeviceProduct, DeviceAction,        SourceIP, SourcePort, DestinationIP, DestinationPort,        Protocol, Activity, Message,        SentBytes = todouble(SentBytes), ReceivedBytes = todouble(ReceivedBytes)    | order by TimeGenerated desc    | take 200    """    csl_df = qry_prov.exec_query(csl_query)    if csl_df is not None and not csl_df.empty:        display(Markdown("### Firewall/Network Events for SpyCloud IPs"))        display(Markdown(f"**Total events:** {len(csl_df)}"))        vendor_summary = csl_df.groupby(['DeviceVendor', 'DeviceAction']).size().reset_index(name='Count')        display(vendor_summary)        display(csl_df.head(30))    else:        print("No CommonSecurityLog events found for tracked IPs.")

In [ ]:
# Check DNS resolution for infected hostsif TARGET_DEVICES or TARGET_IPS:    dns_where = []    if TARGET_DEVICES:        dns_where.extend([f'Computer contains "{d}"' for d in TARGET_DEVICES])    if TARGET_IPS:        dns_where.extend([f'IPAddresses contains "{ip}"' for ip in TARGET_IPS])    dns_query = f"""    DnsEvents    | where TimeGenerated >= ago(7d)    | where {" or ".join(dns_where)}    | project        TimeGenerated, Computer, Name, QueryType, Result,        IPAddresses, ClientIP, SubType    | summarize        QueryCount = count(),        FirstSeen = min(TimeGenerated),        LastSeen = max(TimeGenerated)      by Name, QueryType, Computer    | order by QueryCount desc    | take 50    """    dns_df = qry_prov.exec_query(dns_query)    if dns_df is not None and not dns_df.empty:        display(Markdown("### DNS Activity for Infected Hosts"))        display(dns_df)        fig = px.bar(            dns_df.head(20), x='Name', y='QueryCount',            title="Top DNS Queries from Infected Devices",            color='Computer'        )        fig.update_layout(xaxis_tickangle=-45)        fig.show()    else:        print("No DNS events found.")

In [ ]:
# Fortinet FSSO session lookupif TARGET_USER or TARGET_IPS:    where_parts = []    if TARGET_USER:        where_parts.append(f'user_s contains "{TARGET_USER}"')    if TARGET_IPS:        ip_parts = " or ".join([f'srcip_s == "{ip}" or dstip_s == "{ip}"' for ip in TARGET_IPS])        where_parts.append(f"({ip_parts})")    fortinet_query = f"""    CommonSecurityLog    | where DeviceVendor == "Fortinet"    | where TimeGenerated >= ago(30d)    | where {" or ".join(where_parts)}    | project        TimeGenerated, Activity, SourceIP,        DestinationIP, DestinationPort,        DeviceAction, ApplicationProtocol,        SourceUserName, Message    | order by TimeGenerated desc    | take 100    """    fortinet_df = qry_prov.exec_query(fortinet_query)    if fortinet_df is not None and not fortinet_df.empty:        display(Markdown("### Fortinet FSSO Sessions"))        display(fortinet_df)    else:        print("No Fortinet FSSO sessions found.")

In [ ]:
# PaloAlto User-ID traffic analysisif TARGET_USER or TARGET_IPS:    where_parts = []    if TARGET_USER:        where_parts.append(f'SourceUserName contains "{TARGET_USER}"')    if TARGET_IPS:        ip_parts = " or ".join([f'SourceIP == "{ip}" or DestinationIP == "{ip}"' for ip in TARGET_IPS])        where_parts.append(f"({ip_parts})")    palo_query = f"""    CommonSecurityLog    | where DeviceVendor == "Palo Alto Networks"    | where TimeGenerated >= ago(30d)    | where {" or ".join(where_parts)}    | project        TimeGenerated, Activity, DeviceAction,        SourceIP, DestinationIP, DestinationPort,        ApplicationProtocol, SourceUserName,        DeviceCustomString1Label, DeviceCustomString1    | summarize EventCount = count() by DestinationIP, DestinationPort, DeviceAction    | order by EventCount desc    | take 50    """    palo_df = qry_prov.exec_query(palo_query)    if palo_df is not None and not palo_df.empty:        display(Markdown("### Palo Alto Traffic Analysis"))        display(palo_df)    else:        print("No Palo Alto traffic found.")

In [ ]:
# Azure Firewall correlationif TARGET_IPS:    ip_filter = " or ".join([f'SourceIp == "{ip}" or DestinationIp == "{ip}"' for ip in TARGET_IPS])    azfw_query = f"""    AzureDiagnostics    | where Category == "AzureFirewallNetworkRule" or Category == "AzureFirewallApplicationRule"    | where TimeGenerated >= ago(30d)    | where {ip_filter}    | project        TimeGenerated, Category, OperationName,        msg_s, SourceIp = extract("from ([\\d.]+):", 1, msg_s),        DestinationIp = extract("to ([\\d.]+):", 1, msg_s),        Action = extract("Action: (\\w+)", 1, msg_s),        Protocol = extract("(TCP|UDP|ICMP)", 1, msg_s)    | order by TimeGenerated desc    | take 100    """    azfw_df = qry_prov.exec_query(azfw_query)    if azfw_df is not None and not azfw_df.empty:        display(Markdown("### Azure Firewall Events"))        display(azfw_df)    else:        print("No Azure Firewall events found for tracked IPs.")

In [ ]:
# Network graph visualizationif csl_df is not None and not csl_df.empty:    G = nx.DiGraph()    # Add nodes and edges from firewall data    for _, row in csl_df.head(50).iterrows():        src = row.get('SourceIP', 'unknown')        dst = row.get('DestinationIP', 'unknown')        action = row.get('DeviceAction', 'unknown')        G.add_node(src, node_type='source_ip')        G.add_node(dst, node_type='dest_ip')        G.add_edge(src, dst, action=action)    # Mark SpyCloud IPs    for ip in TARGET_IPS:        if ip in G.nodes:            G.nodes[ip]['node_type'] = 'spycloud_ip'    pos = nx.spring_layout(G, k=2, iterations=50)    edge_x, edge_y = [], []    for edge in G.edges():        x0, y0 = pos[edge[0]]        x1, y1 = pos[edge[1]]        edge_x.extend([x0, x1, None])        edge_y.extend([y0, y1, None])    edge_trace = go.Scatter(x=edge_x, y=edge_y, mode='lines',                            line=dict(width=0.5, color='#888'),                            hoverinfo='none')    node_x = [pos[n][0] for n in G.nodes()]    node_y = [pos[n][1] for n in G.nodes()]    node_colors = ['red' if G.nodes[n].get('node_type') == 'spycloud_ip'                   else 'blue' for n in G.nodes()]    node_text = list(G.nodes())    node_trace = go.Scatter(        x=node_x, y=node_y, mode='markers+text',        text=node_text, textposition='top center', textfont_size=8,        marker=dict(size=10, color=node_colors, line_width=1),        hoverinfo='text'    )    fig = go.Figure(data=[edge_trace, node_trace],                    layout=go.Layout(                        title="Network Communication Graph",                        showlegend=False, height=600,                        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),                        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False)                    ))    fig.show()else:    print("No network data available for graph visualization.")

## Section 7: O365 Activity AnalysisExamine Office 365 activity for the compromised user to detect post-compromise lateral movement or data exfiltration.

In [ ]:
# OfficeActivity for affected userif TARGET_USER:    o365_query = f"""    OfficeActivity    | where TimeGenerated >= ago(30d)    | where UserId contains "{TARGET_USER}"    | summarize EventCount = count() by Operation, OfficeWorkload    | order by EventCount desc    | take 50    """    o365_df = qry_prov.exec_query(o365_query)    if o365_df is not None and not o365_df.empty:        display(Markdown("### O365 Activity Summary"))        display(o365_df)        fig = px.treemap(            o365_df, path=['OfficeWorkload', 'Operation'], values='EventCount',            title="O365 Activity Treemap"        )        fig.show()    else:        print("No O365 activity found.")

In [ ]:
# Mail forwarding rulesif TARGET_USER:    fwd_query = f"""    OfficeActivity    | where TimeGenerated >= ago(30d)    | where UserId contains "{TARGET_USER}"    | where Operation in ("New-InboxRule", "Set-InboxRule", "Set-Mailbox",                          "New-TransportRule", "Set-TransportRule",                          "UpdateInboxRules", "Set-MailboxJunkEmailConfiguration")    | project        TimeGenerated, Operation, UserId,        Parameters = tostring(Parameters),        ClientIP, ResultStatus    | order by TimeGenerated desc    """    fwd_df = qry_prov.exec_query(fwd_query)    if fwd_df is not None and not fwd_df.empty:        display(Markdown("### Mail Forwarding Rules (CRITICAL - Check for Exfiltration)"))        display(fwd_df)    else:        print("No mail forwarding rule changes found.")

In [ ]:
# SharePoint access patternsif TARGET_USER:    sp_query = f"""    OfficeActivity    | where TimeGenerated >= ago(30d)    | where OfficeWorkload == "SharePoint"    | where UserId contains "{TARGET_USER}"    | where Operation in ("FileDownloaded", "FileUploaded", "FileAccessed",                          "FileModified", "FileSyncDownloadedFull",                          "SharingSet", "SharingInvitationCreated",                          "AnonymousLinkCreated")    | summarize EventCount = count() by Operation, Site_Url    | order by EventCount desc    | take 30    """    sp_df = qry_prov.exec_query(sp_query)    if sp_df is not None and not sp_df.empty:        display(Markdown("### SharePoint Access Patterns"))        display(sp_df)    else:        print("No notable SharePoint activity found.")

In [ ]:
# OneDrive file sharingif TARGET_USER:    od_query = f"""    OfficeActivity    | where TimeGenerated >= ago(30d)    | where OfficeWorkload == "OneDrive"    | where UserId contains "{TARGET_USER}"    | where Operation has_any ("Sharing", "Anonymous", "FileDownloaded")    | project        TimeGenerated, Operation, SourceFileName,        Site_Url, ClientIP, ResultStatus    | order by TimeGenerated desc    """    od_df = qry_prov.exec_query(od_query)    if od_df is not None and not od_df.empty:        display(Markdown("### OneDrive File Sharing Activity"))        display(od_df)    else:        print("No OneDrive sharing activity found.")

In [ ]:
# Teams external sharingif TARGET_USER:    teams_query = f"""    OfficeActivity    | where TimeGenerated >= ago(30d)    | where OfficeWorkload == "MicrosoftTeams"    | where UserId contains "{TARGET_USER}"    | where Operation has_any ("MemberAdded", "FileDownloaded",                               "ChatCreated", "MessageSent")    | project        TimeGenerated, Operation, UserId,        Members = tostring(Members),        ChatName, ChannelName, TeamName, CommunicationType    | order by TimeGenerated desc    | take 50    """    teams_df = qry_prov.exec_query(teams_query)    if teams_df is not None and not teams_df.empty:        display(Markdown("### Teams Activity"))        display(teams_df)    else:        print("No notable Teams activity found.")

In [ ]:
# eDiscovery exportsif TARGET_USER:    ediscovery_query = f"""    OfficeActivity    | where TimeGenerated >= ago(30d)    | where Operation has_any ("SearchStarted", "SearchExported",                               "CaseCreated", "HoldCreated",                               "ComplianceSearchExported")    | where UserId contains "{TARGET_USER}" or      Parameters has "{TARGET_USER}"    | project        TimeGenerated, Operation, UserId,        Parameters = tostring(Parameters), ResultStatus    | order by TimeGenerated desc    """    ediscovery_df = qry_prov.exec_query(ediscovery_query)    if ediscovery_df is not None and not ediscovery_df.empty:        display(Markdown("### eDiscovery Activity (CRITICAL - Potential Data Exfiltration)"))        display(ediscovery_df)    else:        print("No eDiscovery activity found.")

## Section 8: UEBA & Behavioral AnalysisAnalyze user and entity behavior analytics to identify anomalous activity patterns that may indicate account compromise.

In [ ]:
# BehaviorAnalytics for userif TARGET_USER:    ueba_query = f"""    BehaviorAnalytics    | where TimeGenerated >= ago(30d)    | where UserPrincipalName contains "{TARGET_USER}"    | project        TimeGenerated,        UserPrincipalName,        ActionType,        ActivityType,        InvestigationPriority,        UsersInsights = tostring(UsersInsights),        DevicesInsights = tostring(DevicesInsights),        ActivityInsights = tostring(ActivityInsights),        SourceIPAddress,        SourceDevice    | order by TimeGenerated desc    """    ueba_df = qry_prov.exec_query(ueba_query)    if ueba_df is not None and not ueba_df.empty:        display(Markdown("### UEBA Behavior Analytics"))        display(Markdown(f"**Total behavior records:** {len(ueba_df)}"))        display(ueba_df.head(20))    else:        print("No UEBA data found.")

In [ ]:
# Anomaly scores over timeif ueba_df is not None and not ueba_df.empty:    ueba_ts = ueba_df.copy()    ueba_ts['TimeGenerated'] = pd.to_datetime(ueba_ts['TimeGenerated'])    ueba_ts['InvestigationPriority'] = pd.to_numeric(        ueba_ts['InvestigationPriority'], errors='coerce'    )    daily_scores = ueba_ts.groupby(        ueba_ts['TimeGenerated'].dt.date    ).agg(        AvgPriority=('InvestigationPriority', 'mean'),        MaxPriority=('InvestigationPriority', 'max'),        EventCount=('TimeGenerated', 'count')    ).reset_index()    daily_scores.columns = ['Date', 'AvgPriority', 'MaxPriority', 'EventCount']    fig = make_subplots(specs=[[{"secondary_y": True}]])    fig.add_trace(        go.Scatter(x=daily_scores['Date'], y=daily_scores['MaxPriority'],                   name='Max Priority Score', line=dict(color='red')),        secondary_y=False    )    fig.add_trace(        go.Bar(x=daily_scores['Date'], y=daily_scores['EventCount'],               name='Event Count', opacity=0.3),        secondary_y=True    )    fig.update_layout(title="UEBA Anomaly Scores Over Time", height=400)    fig.update_yaxes(title_text="Investigation Priority", secondary_y=False)    fig.update_yaxes(title_text="Event Count", secondary_y=True)    fig.show()

In [ ]:
# Peer group comparisonif TARGET_USER:    peer_query = f"""    BehaviorAnalytics    | where TimeGenerated >= ago(30d)    | extend UserDept = tostring(parse_json(UsersInsights).UserDepartment)    | where UserDept != ""    | summarize        AvgPriority = avg(InvestigationPriority),        MaxPriority = max(InvestigationPriority),        EventCount = count()      by UserPrincipalName, UserDept    | extend IsTarget = iff(UserPrincipalName contains "{TARGET_USER}", true, false)    | order by AvgPriority desc    """    peer_df = qry_prov.exec_query(peer_query)    if peer_df is not None and not peer_df.empty:        display(Markdown("### Peer Group Comparison"))        fig = px.scatter(            peer_df, x='EventCount', y='AvgPriority',            color='IsTarget', size='MaxPriority',            hover_data=['UserPrincipalName'],            title="User vs Peer Group: Investigation Priority",            color_discrete_map={True: 'red', False: 'lightblue'}        )        fig.show()    else:        print("No peer group data available.")

In [ ]:
# First-time eventsif TARGET_USER:    first_time_query = f"""    BehaviorAnalytics    | where TimeGenerated >= ago(30d)    | where UserPrincipalName contains "{TARGET_USER}"    | extend Insights = parse_json(ActivityInsights)    | where Insights.FirstTimeUserConnectedViaISP == true        or Insights.FirstTimeUserAccessedApplication == true        or Insights.FirstTimeConnectionFromCountry == true        or Insights.FirstTimeUserLoggedOnToDevice == true    | project        TimeGenerated, ActionType, ActivityType,        FirstTimeISP = tobool(Insights.FirstTimeUserConnectedViaISP),        FirstTimeApp = tobool(Insights.FirstTimeUserAccessedApplication),        FirstTimeCountry = tobool(Insights.FirstTimeConnectionFromCountry),        FirstTimeDevice = tobool(Insights.FirstTimeUserLoggedOnToDevice),        SourceIPAddress, SourceDevice    | order by TimeGenerated desc    """    first_df = qry_prov.exec_query(first_time_query)    if first_df is not None and not first_df.empty:        display(Markdown("### First-Time Events (Anomalous)"))        display(first_df)    else:        print("No first-time events detected.")

In [ ]:
# Risk score timelineif ueba_df is not None and not ueba_df.empty:    risk_ts = ueba_df.copy()    risk_ts['TimeGenerated'] = pd.to_datetime(risk_ts['TimeGenerated'])    risk_ts['InvestigationPriority'] = pd.to_numeric(        risk_ts['InvestigationPriority'], errors='coerce'    )    # Calculate rolling risk score    risk_ts = risk_ts.sort_values('TimeGenerated')    risk_ts['RollingRisk'] = risk_ts['InvestigationPriority'].rolling(        window=5, min_periods=1    ).mean()    fig = go.Figure()    fig.add_trace(go.Scatter(        x=risk_ts['TimeGenerated'], y=risk_ts['InvestigationPriority'],        mode='markers', name='Raw Priority', marker=dict(size=6, opacity=0.5)    ))    fig.add_trace(go.Scatter(        x=risk_ts['TimeGenerated'], y=risk_ts['RollingRisk'],        mode='lines', name='Rolling Average (5)',        line=dict(color='red', width=2)    ))    fig.add_hline(y=5, line_dash="dash", line_color="orange",                  annotation_text="Warning Threshold")    fig.add_hline(y=8, line_dash="dash", line_color="red",                  annotation_text="Critical Threshold")    fig.update_layout(title="UEBA Risk Score Timeline", height=400,                      xaxis_title="Time", yaxis_title="Investigation Priority")    fig.show()

## Section 9: Remediation StatusTrack the status of automated and manual remediation actions taken in response to the SpyCloud exposure.

In [ ]:
# Check SpyCloud_ConditionalAccessLogs_CLif TARGET_USER:    ca_log_query = f"""    SpyCloud_ConditionalAccessLogs_CL    | where UserPrincipalName_s contains "{TARGET_USER}"    | project        TimeGenerated,        UserPrincipalName_s,        PolicyName_s,        GrantControls_s,        SessionControls_s,        Result_s,        CorrelationId_s,        AppDisplayName_s,        DevicePlatform_s,        IPAddress_s    | order by TimeGenerated desc    """    ca_log_df = qry_prov.exec_query(ca_log_query)    if ca_log_df is not None and not ca_log_df.empty:        display(Markdown("### Conditional Access Enforcement Logs"))        display(ca_log_df)    else:        print("No CA enforcement logs found.")

In [ ]:
# Check Spycloud_MDE_Logs_CL remediation actionsif TARGET_USER or TARGET_DEVICES:    where_parts = []    if TARGET_USER:        where_parts.append(f'AccountName_s contains "{TARGET_USER}"')    if TARGET_DEVICES:        dev_parts = " or ".join([f'DeviceName_s contains "{d}"' for d in TARGET_DEVICES])        where_parts.append(f"({dev_parts})")    mde_rem_query = f"""    Spycloud_MDE_Logs_CL    | where {" or ".join(where_parts)}    | where ActionType_s has_any ("Isolate", "Remediate", "Scan",                                   "Restrict", "StopAndQuarantine")    | project        TimeGenerated,        DeviceName_s,        AccountName_s,        ActionType_s,        RemediationStatus_s,        ActionStatus,        ScanStatus_s,        ThreatName_s    | order by TimeGenerated desc    """    mde_rem_df = qry_prov.exec_query(mde_rem_query)    if mde_rem_df is not None and not mde_rem_df.empty:        display(Markdown("### MDE Remediation Actions"))        display(mde_rem_df)    else:        print("No MDE remediation actions found.")

In [ ]:
# Password reset statusif TARGET_USER:    pwd_reset_query = f"""    AuditLogs    | where TimeGenerated >= ago(30d)    | where OperationName has_any (        "Reset password", "Change password",        "Reset user password", "Change user password"    )    | extend Target = tostring(TargetResources[0].userPrincipalName)    | where Target contains "{TARGET_USER}"    | project        TimeGenerated,        OperationName,        Target,        InitiatedBy = coalesce(            tostring(InitiatedBy.user.userPrincipalName),            tostring(InitiatedBy.app.displayName)        ),        Result,        ResultReason    | order by TimeGenerated desc    """    pwd_df = qry_prov.exec_query(pwd_reset_query)    if pwd_df is not None and not pwd_df.empty:        display(Markdown("### Password Reset History"))        display(pwd_df)    else:        display(Markdown("### Password Reset History"))        display(Markdown("**WARNING: No password reset detected since exposure. Immediate action required.**"))

In [ ]:
# MFA re-registration statusif TARGET_USER:    mfa_reg_query = f"""    AuditLogs    | where TimeGenerated >= ago(30d)    | where OperationName has_any (        "User registered security info",        "Admin registered security info",        "User deleted security info"    )    | extend Target = tostring(TargetResources[0].userPrincipalName)    | where Target contains "{TARGET_USER}"    | project TimeGenerated, OperationName, Target,              InitiatedBy = tostring(InitiatedBy.user.userPrincipalName),              Result    | order by TimeGenerated desc    """    mfa_reg_df = qry_prov.exec_query(mfa_reg_query)    if mfa_reg_df is not None and not mfa_reg_df.empty:        display(Markdown("### MFA Re-registration Status"))        display(mfa_reg_df)    else:        display(Markdown("### MFA Re-registration Status"))        display(Markdown("**No MFA re-registration detected. Consider requiring re-registration.**"))

In [ ]:
# Security group changesif TARGET_USER:    group_query = f"""    AuditLogs    | where TimeGenerated >= ago(30d)    | where OperationName has_any (        "Add member to group", "Remove member from group",        "Add member to role", "Remove member from role"    )    | mv-expand TargetResources    | extend Target = tostring(TargetResources.userPrincipalName)    | where Target contains "{TARGET_USER}"    | project TimeGenerated, OperationName,              GroupName = tostring(TargetResources.displayName),              Target,              InitiatedBy = tostring(InitiatedBy.user.userPrincipalName),              Result    | order by TimeGenerated desc    """    group_df = qry_prov.exec_query(group_query)    if group_df is not None and not group_df.empty:        display(Markdown("### Security Group Changes"))        display(group_df)    else:        print("No security group changes found.")

In [ ]:
# CA policy enforcement status - summary dashboardremediation_summary = {    "Password Reset": "COMPLETED" if (pwd_df is not None and not pwd_df.empty) else "PENDING",    "MFA Re-registration": "COMPLETED" if (mfa_reg_df is not None and not mfa_reg_df.empty) else "PENDING",    "CA Policy Enforcement": "ACTIVE" if (ca_log_df is not None and not ca_log_df.empty) else "NOT DETECTED",    "Device Isolation": "COMPLETED" if (mde_rem_df is not None and not mde_rem_df.empty) else "PENDING",    "Group Membership Review": "COMPLETED" if (group_df is not None and not group_df.empty) else "PENDING",}display(Markdown("### Remediation Status Dashboard"))for action, status in remediation_summary.items():    icon = "✅" if status == "COMPLETED" or status == "ACTIVE" else "❌"    display(Markdown(f"- {icon} **{action}:** {status}"))

## Section 10: Recommendations & ActionsAuto-generated recommendations based on the investigation findings, with a calculated risk score and remediation checklist.

In [ ]:
# Auto-generated recommendations based on findingsdef calculate_risk_score():    """Calculate composite risk score (0-100) based on investigation findings."""    score = 0    factors = {}    # Factor 1: Exposure count (0-20 points)    if breach_df is not None and not breach_df.empty:        exposure_count = len(breach_df)        exposure_score = min(20, exposure_count * 2)        score += exposure_score        factors['Exposure Count'] = (exposure_score, f"{exposure_count} exposures")    # Factor 2: Plaintext passwords (0-20 points)    if breach_df is not None and not breach_df.empty:        plaintext = breach_df[breach_df['password_type_s'] == 'plaintext']        pct = len(plaintext) / len(breach_df) * 100 if len(breach_df) > 0 else 0        pt_score = min(20, int(pct / 5))        score += pt_score        factors['Plaintext Passwords'] = (pt_score, f"{pct:.1f}%")    # Factor 3: Risky sign-ins (0-15 points)    if 'risk_df' in dir() and risk_df is not None and not risk_df.empty:        risk_count = len(risk_df)        risk_score = min(15, risk_count * 3)        score += risk_score        factors['Risky Sign-ins'] = (risk_score, f"{risk_count} events")    # Factor 4: UEBA anomalies (0-15 points)    if ueba_df is not None and not ueba_df.empty:        high_priority = ueba_df[pd.to_numeric(ueba_df['InvestigationPriority'], errors='coerce') > 5]        ueba_score = min(15, len(high_priority) * 3)        score += ueba_score        factors['UEBA High Priority'] = (ueba_score, f"{len(high_priority)} events")    # Factor 5: Remediation gap (0-15 points)    pending = sum(1 for s in remediation_summary.values() if s == "PENDING")    rem_score = pending * 5    score += rem_score    factors['Pending Remediations'] = (rem_score, f"{pending} actions pending")    # Factor 6: Mail forwarding rules (0-15 points)    if 'fwd_df' in dir() and fwd_df is not None and not fwd_df.empty:        fwd_score = 15        score += fwd_score        factors['Mail Forwarding Rules'] = (fwd_score, "DETECTED")    return min(100, score), factorsrisk_score, risk_factors = calculate_risk_score()# Display risk scoredisplay(Markdown("### Investigation Risk Score"))color = "red" if risk_score >= 70 else "orange" if risk_score >= 40 else "green"level = "CRITICAL" if risk_score >= 70 else "HIGH" if risk_score >= 40 else "MODERATE"display(HTML(f'<h2 style="color: {color};">Risk Score: {risk_score}/100 ({level})</h2>'))display(Markdown("#### Risk Factor Breakdown"))for factor, (pts, detail) in risk_factors.items():    display(Markdown(f"- **{factor}:** {pts} pts ({detail})"))

In [ ]:
# Suggested remediation actions checklistdisplay(Markdown("### Remediation Actions Checklist"))recommendations = []if remediation_summary.get("Password Reset") == "PENDING":    recommendations.append("🔴 **IMMEDIATE:** Force password reset for the affected user")if remediation_summary.get("MFA Re-registration") == "PENDING":    recommendations.append("🔴 **IMMEDIATE:** Revoke existing MFA methods and require re-registration")if remediation_summary.get("Device Isolation") == "PENDING" and TARGET_DEVICES:    recommendations.append("🔴 **IMMEDIATE:** Isolate infected devices via MDE")if 'fwd_df' in dir() and fwd_df is not None and not fwd_df.empty:    recommendations.append("🔴 **IMMEDIATE:** Review and remove suspicious mail forwarding rules")if breach_df is not None and not breach_df.empty:    plaintext = breach_df[breach_df['password_type_s'] == 'plaintext']    if not plaintext.empty:        recommendations.append("🟠 **HIGH:** Exposed plaintext passwords detected - check for password reuse across systems")if 'risk_df' in dir() and risk_df is not None and not risk_df.empty:    recommendations.append("🟠 **HIGH:** Review risky sign-in events and revoke active sessions")recommendations.extend([    "🟡 **MEDIUM:** Review OAuth app consents for the affected user",    "🟡 **MEDIUM:** Check for lateral movement from compromised account",    "🟢 **LOW:** Update breach notification records",    "🟢 **LOW:** Schedule user security awareness training",])for i, rec in enumerate(recommendations, 1):    display(Markdown(f"{i}. {rec}"))

In [ ]:
# Export investigation reportreport_data = {    "incident_id": INCIDENT_ID,    "target_user": TARGET_USER,    "target_devices": TARGET_DEVICES,    "target_ips": TARGET_IPS,    "risk_score": risk_score,    "risk_level": level,    "risk_factors": {k: {"score": v[0], "detail": v[1]} for k, v in risk_factors.items()},    "remediation_status": remediation_summary,    "exposure_count": len(breach_df) if breach_df is not None else 0,    "signin_count": len(signin_df) if signin_df is not None else 0,    "timestamp": datetime.utcnow().isoformat(),}import jsonreport_json = json.dumps(report_data, indent=2, default=str)# Save reportreport_path = f"spycloud_investigation_{INCIDENT_ID}_{datetime.utcnow().strftime('%Y%m%d_%H%M%S')}.json"with open(report_path, 'w') as f:    f.write(report_json)display(Markdown(f"### Investigation report exported to: `{report_path}`"))print(report_json)

## Section 11: Investigation GraphInteractive network graph visualizing the relationships between the user, credentials, breaches, devices, IPs, sign-in locations, applications, and network events.

In [ ]:
# Build comprehensive investigation graphG = nx.Graph()# Central node: Userif TARGET_USER:    G.add_node(TARGET_USER, node_type='user', size=30)    # User -> Exposed Credentials -> Breach Sources    if breach_df is not None and not breach_df.empty:        for _, row in breach_df.iterrows():            cred_id = f"cred_{row.get('document_id_s', 'unknown')[:8]}"            breach_title = str(row.get('breach_title_s', 'Unknown Breach'))[:30]            G.add_node(cred_id, node_type='credential',                       size=10, label=row.get('password_type_s', ''))            G.add_node(breach_title, node_type='breach', size=15)            G.add_edge(TARGET_USER, cred_id, relation='exposed_credential')            G.add_edge(cred_id, breach_title, relation='from_breach')    # User -> Devices -> IP Addresses    for device in TARGET_DEVICES:        G.add_node(device, node_type='device', size=20)        G.add_edge(TARGET_USER, device, relation='uses_device')    for ip in TARGET_IPS:        G.add_node(ip, node_type='ip', size=12)        G.add_edge(TARGET_USER, ip, relation='connected_from')        for device in TARGET_DEVICES:            G.add_edge(device, ip, relation='network_connection')    # User -> Sign-in Locations -> Applications    if signin_df is not None and not signin_df.empty:        locations = signin_df['Location'].dropna().unique()[:10]        for loc in locations:            G.add_node(loc, node_type='location', size=12)            G.add_edge(TARGET_USER, loc, relation='signed_in_from')        apps = signin_df['AppDisplayName'].dropna().unique()[:10]        for app in apps:            G.add_node(app, node_type='application', size=12)            G.add_edge(TARGET_USER, app, relation='accessed_app')    # Device -> Network -> Firewall    if 'csl_df' in dir() and csl_df is not None and not csl_df.empty:        for _, row in csl_df.head(20).iterrows():            dst = row.get('DestinationIP', '')            if dst:                fw_node = f"FW:{row.get('DeviceVendor', 'Unknown')}"                G.add_node(dst, node_type='external_ip', size=8)                G.add_node(fw_node, node_type='firewall', size=15)                G.add_edge(dst, fw_node, relation='firewall_event')# Color mapping by node typecolor_map = {    'user': '#FF4444',    'credential': '#FF8800',    'breach': '#AA0000',    'device': '#4488FF',    'ip': '#44BB44',    'location': '#8844FF',    'application': '#44BBBB',    'external_ip': '#888888',    'firewall': '#FF44FF',}# Layoutpos = nx.spring_layout(G, k=3, iterations=100, seed=42)# Build plotly tracesedge_traces = []for edge in G.edges(data=True):    x0, y0 = pos[edge[0]]    x1, y1 = pos[edge[1]]    edge_traces.append(        go.Scatter(x=[x0, x1, None], y=[y0, y1, None],                   mode='lines', line=dict(width=0.5, color='#cccccc'),                   hoverinfo='none', showlegend=False)    )# Group nodes by type for legendnode_traces = {}for node in G.nodes(data=True):    ntype = node[1].get('node_type', 'unknown')    if ntype not in node_traces:        node_traces[ntype] = {'x': [], 'y': [], 'text': [], 'size': []}    x, y = pos[node[0]]    node_traces[ntype]['x'].append(x)    node_traces[ntype]['y'].append(y)    node_traces[ntype]['text'].append(str(node[0]))    node_traces[ntype]['size'].append(node[1].get('size', 10))fig = go.Figure()for trace in edge_traces:    fig.add_trace(trace)for ntype, data in node_traces.items():    fig.add_trace(go.Scatter(        x=data['x'], y=data['y'],        mode='markers+text',        text=data['text'],        textposition='top center',        textfont=dict(size=7),        marker=dict(            size=data['size'],            color=color_map.get(ntype, '#999999'),            line=dict(width=1, color='white')        ),        name=ntype.replace('_', ' ').title(),        hovertext=data['text'],        hoverinfo='text'    ))fig.update_layout(    title="SpyCloud Incident Investigation Graph",    height=800,    showlegend=True,    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),    plot_bgcolor='white')fig.show()print(f"\nGraph Statistics:")print(f"  Nodes: {G.number_of_nodes()}")print(f"  Edges: {G.number_of_edges()}")print(f"  Node types: {set(nx.get_node_attributes(G, 'node_type').values())}")

---**End of SpyCloud Incident Triage Notebook**For questions or issues, contact the SOC engineering team.